# 06：在量子计算机上测量你的电路

**你将学到：**理想概率描述模型，真实硬件返回有限次测量的计数。一次 shot 是一次执行加一次测量。

我们将三比特非对称 RY 电路提交到夸父（Quafu），并把测量计数与理想预测比较。

[English edition](../notebooks/06_quafu_hardware.ipynb)

**学习目标：** 提交非对称电路，读懂 shots 计数，并在明确位序下与理想概率比较。

**先修要求：** 已完成 01；05 有助于理解任务生命周期。真实执行需要夸父凭据、可用设备和 shots 额度。本地预演不涉及真机。

学习路线见 [workshop 导航](../README.md)。


## 1. 准备 workshop 环境

直接运行设置单元，无需修改。平台凭据会自动读取，不会在这里输入或显示。
本课使用独立的非对称实验结果文件，不覆盖或复用旧 Bell 回执。

中英文版使用同一用户的个人结果目录和回执；换一种语言学习时，仍会恢复已经保存的任务，而不是重新提交。


In [ ]:
from pathlib import Path
import os
import sys
import json

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. 构造电路并检查理想答案

只对 q0 施加 RY(pi/3)。理想概率为 P(000)=0.75、P(100)=0.25，q1 和 q2 始终为零。非对称分布能够暴露位序反转。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import math
import flagquantum as fq

circuit = fq.Circuit(3).ry(0, math.pi / 3)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
ideal = fq.run(circuit)
labels = [format(index, "03b") for index in range(8)]
ideal_probabilities = ideal.to_statevector().reshape(-1).abs().square().tolist()
print("Ideal probabilities (q0 q1 q2):", dict(zip(labels, ideal_probabilities)))


## 3. 选择量子硬件

请和讲师确认当天可用的目标设备。一次 shot 是一次执行并测量。
**打开提交开关并运行提交单元，会创建一个测量 1024 次的真实 QPU 任务，使用你的额度。**

公开示例默认本地预演（`False`）；活动部署须在讲师确认凭据、设备与额度后将可见开关设为 `True`，才能逐格真实提交。下面显示实际目标、shots 和结果目录；不会因为检测到凭据就自动提交。


In [ ]:
target = os.getenv("QUAFU_TARGET", "quafu:Baihua")
shots = 1024
submit_to_hardware = False

print("Target:", target)
print("Shots:", shots)
print("Result directory:", OUTPUTS)
print("Real submission enabled:", submit_to_hardware)


### 确定回执位置（直接运行，无需修改）

此单元在个人持久化结果目录中确定回执位置，并检查旧版 workshop 是否留下状态不明的提交。保存、配置校验和匹配恢复由 SDK 的 `receipt=` 完成，请保留回执及 SDK 创建的配套记录。

旧版手动保存的回执须显式使用 `fq.restore_job(receipt_path)`，不会被自动接管；不要删除旧记录来强行再次提交。


In [ ]:
receipt_path = OUTPUTS / "quafu-ry-q0-three-qubits.receipt.json"
job = None
if submit_to_hardware:
    if not os.getenv("QUAFU_API_KEY"):
        raise RuntimeError("Configure Quafu access outside this notebook; nothing submitted.")
    previous_files = (
        OUTPUTS / "quafu-ry-q0-three-qubits.submission.json",
        OUTPUTS / "quafu-ry-q0-three-qubits.json",
    )
    if not receipt_path.exists() and any(path.exists() for path in previous_files):
        raise RuntimeError("Earlier workshop records exist without a receipt. Keep them and reconcile the task; do not resubmit.")
print("Receipt path:", receipt_path.resolve())


## 4. 通过 FlagQuantum 提交

`fq.submit(..., receipt=receipt_path)` 首次提交并自动保存回执，匹配的重复运行会恢复同一任务而不再次提交；调用返回时不会等待执行完成。

配置不同、记录不完整或恢复失败时会报错，不会自动重提；无需手动计算摘要或调用 `job.save()`。确认旧任务状态后，新的实验应使用新的回执路径。


In [ ]:
job = None
if submit_to_hardware:
    job = fq.submit(
        circuit, target=target, shots=shots, outputs=fq.counts(), compiler=None,
        receipt=receipt_path,
    )
    print("Task ID:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. 查询进度

这里执行一次只读查询。`queued` 表示排队，`running` 表示正在执行，`succeeded` 表示平台报告已完成；结果交付仍可能延迟或暂不可用。

**读懂输出：** `Task ID` 标识任务，不表示计算完成。匹配的重复运行在 SDK 恢复后打印相同 ID。只有取回结果，才能完成本地与远程对照。

**安全重复运行：** 使用同一持久化回执路径重复调用提交，SDK 会校验并恢复原任务。等待时也可只重跑结果单元；不要删除旧记录来重试状态不明的提交。改变电路、target 或 shots 时，先核对旧任务，再为新实验选择新的路径。


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. 短暂等待并读取结果

`job.wait()` 将轮询状态和获取结果组合起来。超时不会停止任务，可以稍后重新运行此单元。
也可以在状态为 `succeeded` 后调用 `job.result()`。
如果远程任务失败，不会用本地结果冒充远程结果。

**概念：** 理想概率描述无噪声模型，counts 是有限次 shots 的观测结果。这里 P(000)=0.75、P(100)=0.25，因此 1024 次理想测量的期望计数为 768 和 256，但不保证实际恰好得到这两个计数。

**预期输出：** 下方逐个比特串对照理想概率、实测计数与实测频率。FlagQuantum 的全寄存器结果按 q0,q1,q2 排列。此前实测的夸父平台界面在测量映射 q0→c0、q1→c1、q2→c2 时，按 c2,c1,c0 显示，因此平台的 001 对应 FlagQuantum 的 100。若测量顺序改变或只测部分比特，应查看实际测量映射，不能盲目反转字符串。FlagQuantum 返回的 counts 已完成规范化，不要再反转一次。

**常见误解：** 即使理想设备也会有有限 shots 引起的随机频率波动。设备噪声还可能改变概率、产生理想情况下不可能的结果。增加 shots 能降低采样不确定性，不能消除系统性的设备误差；单张直方图不能唯一确定各类噪声来源。

**一行输出怎么读（示意，并非真机结果）：** `100 ideal: 0.25 count: 256 frequency: 0.25` 表示 1024 次测量中，256 次得到 q0=1,q1=0,q2=0。真机计数不一定恰为 256。两组柱子都使用概率／频率单位；原始计数单独打印，不混用同一纵轴。


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    counts = result.counts[0]
    assert sum(counts.values()) == shots
    print("Measured counts:", counts)
    print("Total shots:", sum(counts.values()))

    frequencies = [counts.get(label, 0) / shots for label in labels]
    for label, probability, frequency in zip(labels, ideal_probabilities, frequencies):
        print(label, "ideal:", round(probability, 4), "count:", counts.get(label, 0), "frequency:", round(frequency, 4))
    positions = list(range(len(labels)))
    plt.bar([p - 0.2 for p in positions], ideal_probabilities, width=0.4, label="Ideal probability")
    plt.bar([p + 0.2 for p in positions], frequencies, width=0.4, label="Hardware frequency")
    plt.xticks(positions, labels)
    plt.xlabel("Bit string (q0 q1 q2)")
    plt.ylabel("Probability / frequency")
    plt.legend()
    plt.show()


### 保存这次非对称实验的结果（可选）

这个折叠导出单元记录真实的任务 ID 和测量计数。理解或使用 `fq.submit()` 并不需要运行这一步。


In [ ]:
if result is not None:
    record = {
        "source": "live", "status": "completed", "target": target,
        "shots": shots, "task_id": job.id, "counts": counts,
    }
    (OUTPUTS / "quafu-ry-q0-three-qubits.json").write_text(json.dumps(record, indent=2))


## 7. 可选扩展：一次提交三个电路

**用途：** 在同一目标上比较多个实验，而不必逐个管理独立的父作业。常见用途包括扫描不同参数、比较不同电路结构。平台的一个父作业包含三个计算任务；批量提交不保证硬件并行执行，也不等于获得性能加速。

这里分别翻转三个量子比特中的一个。按 FlagQuantum 的 q0,q1,q2 位序，电路 0、1、2 的理想结果依次为 `100`、`010`、`001`。非对称结果有助于识别位序反转或结果顺序错误。提交前先展示电路图。


In [ ]:
batch_circuits = [fq.Circuit(3).x(qubit) for qubit in range(3)]
batch_expected = ["100", "010", "001"]

for index, batch_circuit in enumerate(batch_circuits):
    print("Circuit", index, "ideal bit string (q0 q1 q2):", batch_expected[index])
    fig, ax = batch_circuit.draw(format="mpl", show_all_qubits=True)
    plt.show()


### 先确认配额，再主动开启

`shots=1024` 是**每个电路**的数量，不是三个电路共享：本可选批次额外请求 **3 × 1024 = 3072 shots**。若前面的单电路任务也运行，总共请求 4096 shots。批量示例有独立的可见开关；打开前面的单任务开关不会自动开启本批次。只有确定要提交这三个真实任务时，才设置 `submit_batch = True`。

独立回执将本批次与前面的单电路实验区分开来。同一配置重复运行、重启后运行或切换语言，都会恢复同一个父作业。电路、顺序、target 或 shots 改变时会拒绝复用；状态不明的提交应先核对，不能删除恢复文件后重提。


In [ ]:
batch_shots = 1024
batch_receipt_path = OUTPUTS / "quafu-x-each-three-circuits.receipt.json"
submit_batch = False

batch_job = None
print("Batch receipt:", batch_receipt_path.resolve())
print("Shots per circuit:", batch_shots)
print("Additional requested shots:", len(batch_circuits) * batch_shots)
if submit_batch:
    batch_job = fq.submit(
        batch_circuits, target=target, shots=batch_shots, outputs=fq.counts(),
        receipt=batch_receipt_path, name="workshop asymmetric three-circuit check",
    )
    print("Batch job ID:", batch_job.id)
else:
    print("Optional batch not submitted. Set submit_batch = True to request one job with three tasks.")


### 按提交顺序读取结果

即使平台任务完成或返回的顺序不同，`batch_result.counts[i]` 仍对应 `batch_circuits[i]`。三个电路的理想计数依次为 `{'100': 1024}`、`{'010': 1024}`、`{'001': 1024}`；真实硬件可能引入噪声。返回的比特串已归一到 FlagQuantum 位序，不要再次反转。这是三个不同电路的计算任务，不是对同一个电路测量三次。

排队较久时只需重新运行下方结果单元；超时不会停止或重新提交该批次。


In [ ]:
batch_result = None
if batch_job is not None:
    try:
        batch_result = batch_job.wait(timeout=30)
    except TimeoutError:
        print("Batch is still pending. Rerun this result cell later; do not create a new receipt.")

if batch_result is not None:
    assert len(batch_result.counts) == len(batch_circuits)
    for index, batch_counts in enumerate(batch_result.counts):
        assert sum(batch_counts.values()) == batch_shots
        print("Circuit", index, "ideal:", batch_expected[index], "counts:", batch_counts)


## 带回家使用

离开 Jupyter 后仍然使用相同的标准 API。首次调用提交真实任务并保存回执，使用同一持久化路径重复运行会恢复它。SDK 负责配置校验与防重复提交；请先在代码之外配置访问凭据。

```python
import math
import flagquantum as fq

circuit = fq.Circuit(3).ry(0, math.pi / 3)
job = fq.submit(
    circuit, target="quafu:Baihua", shots=1024, outputs=fq.counts(),
    receipt="ry-q0-three-qubits.receipt.json",
)
print(job.id, job.status())
```

恢复已保存任务：`job = fq.restore_job("ry-q0-three-qubits.receipt.json")`。完成后可查看 `job.result().counts`。

**解释结果：**为什么可能出现 `001` 或 `110`？增加 shots 能消除设备噪声吗？
旧 Bell 结果属于不同实验，不能与本课的理想分布直接比较。环境配置和恢复细节见[云任务说明](../CLOUD_JOBS.md)。
